# Applied Algorithms – Tutorial 3
### Solutions will be discussed on May 7
### Form link: https://forms.gle/ePqH1y8aGRw37zcH8

In [1]:
# simple test() function to check function returns vs. what the function was supposed to return.
def test(got, expected):
    if got == expected:
        prefix = ' OK '
    else:
        prefix = '  X '
    print('%s got: %s expected: %s' % (prefix, repr(got), repr(expected)))

## Assignment 1 

1.  Consider the permutation $\pi = 3 \ 4 \ 6 \ 5 \ 8 \ 1 \ 7 \ 2$.
    Expand the permutation according to the lectures and apply $\mathit{BreakpointReversalSort}$ to the extended permutation. Specify all intermediate steps.
2. Now apply $\mathit{BreakpointReversalSort}$ to the (already extended) permutation $\pi = 0 \ 1 \ 5 \ 6 \ 7 \ 2 \ 3 \ 4 \ 8 \ 9$. What do you observe?
3. Apply $\mathit{ImprovedBreakpointReversalSort}$ to the above permutation. Specify all intermediate steps.

1.  
<pre>
0 3 4 6 5 8 1 7 2 9$  
0,3 4,6 5,8,1,7,2,9$   7BP, reverse 1 7  
0,3 4,6 5,8 7,1 2,9$   5BP, reverse 8 7,1 2  
0,3 4,6 5,2 1,7 8 9$   4BP, reverse 3 4,6 5,2 1  
0 1 2,5 6,4 3,7 8 9$   3BP, reverse 5 6,4 3  
0 1 2 3 4,6 5,7 8 9$   2BP, reverse 6 5  
0 1 2 3 4 5 6 7 8 9$   0BP
</pre>

2.  
<pre>
0 1,5 6 7,2 3 4,8 9   3BP, reverse 5 6 7,2 3 4  
0 1,4 3 2,7 6 5,8 9   3BP, reverse 4 3 2  --  no improvement  
0 1 2 3 4,7 6 5,8 9   2BP, reverse 7 6 5  
0 1 2 3 4 5 6 7 8 9   0BP
</pre>



3.  
<pre>
0 1,5 6 7,2 3 4,8 9   3BP, all strips increasing - flip 5 6 7   
0 1,7 6 5,2 3 4,8 9   3BP, reverse 7 6 5,2 3 4  
0 1,4 3 2,5 6 7 8 9   2BP, reverse 4 3 2  
0 1 2 3 4 5 6 7 8 9   0BP
</pre>

## Assignment 2 

In lecture, you studied two pancake sorting algorithms: a Greedy $\mathit{SimpleReversalSort}$ algorithm, for which you have seen a worst-case approximation ratio of (n−1)/2, and $\mathit{ImprovedBreakpointReversalSort}$, which achieves a 4-approximation guarantee.

However, worst-case guarantees do not always reflect how algorithms behave in practice.

In this assignment, you will implement both algorithms and compare their performance on randomly generated permutations of sizes {$5$, $10$, $25$}. For each input size, generate a large number of random permutations, run both algorithms on each instance, and record the number of flips used.

Report the average number of flips for each algorithm, as well as the fraction of instances in which ImprovedBreakpointReversalSort performs better than the greedy algorithm.

In [ ]:
import random
import traceback

def is_identity(sequence):
    return all(sequence[i] == i for i in range(len(sequence)))

def flip_subsequence(sequence, i , j):
    if j < i:
        x = i
        i = j
        j = x
    sequence[i:j+1] = sequence[i:j+1][::-1]

def count_breakpoints(sequence):
    count = 0
    for i in range(1, len(sequence)):
        if sequence[i] - sequence[i - 1] != 1:
            count += 1
    return count

def get_smallest_dec_strip(sequence):
    smallest = 0
    smallest_index = -1
    for i in range(1, len(sequence) - 1):
        diff_left = sequence[i] - sequence[i-1]
        diff_right = sequence[i+1] - sequence[i]
        is_desc = False
        # Single letter is defined as descending
        if abs(diff_left) != 1 and abs(diff_right) != 1:
            is_desc = True
        if diff_left == -1:
            is_desc = True
        if not is_desc:
            continue

        # Set if first found
        if smallest_index == -1:
            #print(f"initial: {sequence[i]}")
            smallest = sequence[i]
            smallest_index = i
            continue
        # Smaller one exists
        if sequence[i] > smallest:
            continue
        smallest = sequence[i]
        smallest_index = i
        #print(f"update: {sequence[i]}")


    return smallest, smallest_index

def get_first_inc_strip(sequence):
    still_start = True
    start = 0
    for i in range(1, len(sequence) - 1):
        still_start = still_start and sequence[i] == sequence[i-1] + 1
        if still_start:
            continue
        inc = sequence[i+1] == sequence[i] + 1
        if inc: 
            start = i
            break

    for i in range(start, len(sequence) - 1):
        if sequence[i+1] != sequence[i] + 1:
            return start, i




def simple_reversal_sort(sequence):
    flips = 0
    for i in range(len(sequence)):
        index = sequence.index(i)
        if index != i:
            flip_subsequence(sequence, i, index)
            flips += 1
        if is_identity(sequence):
            return flips
    
    return flips

def breakpoint_reversal_sort(sequence):
    flips = 0
    sequence.append(max(sequence) + 1)
    sequence.insert(0, min(sequence) - 1)

    while count_breakpoints(sequence) > 0:
        # todo: best, not just one!
        smallest, smallest_index = get_smallest_dec_strip(sequence)
        
        # No decreasing found:
        if smallest_index == -1:
            flip_start, flip_end = get_first_inc_strip(sequence)
            flip_subsequence(sequence, flip_start, flip_end)
            flips += 1
        else:
            k = smallest_index
            k_min_one = sequence.index(smallest - 1)
            if k_min_one > k:
                k_min_one -= 1
            else:
                k_min_one += 1
            flip_subsequence(sequence, k_min_one, k)
            flips += 1
    return flips


num_tests = 100
perm_sizes = [5, 10, 25]
for size in perm_sizes:
    bbetter = 0
    totalb = 0
    totals = 0
    for i in range(num_tests):
        rand_seq = random.sample(range(0, size), size)
        rand_seq2 = rand_seq.copy()
        bsteps = breakpoint_reversal_sort(rand_seq)
        ssteps = simple_reversal_sort(rand_seq2)
        totalb += bsteps
        totals += ssteps
        if bsteps < ssteps:
            bbetter += 1
    print(f"{size}: Breakpoints Better: {bbetter / num_tests}")
    print(f"{size}: AVG Simple: {totals / num_tests}")
    print(f"{size}: AVG Breakpoint: {totalb / num_tests}")
    print()


5: Breakpoints Better: 0.0
5: AVG Simple: 2.79
5: AVG Breakpoint: 2.94

10: Breakpoints Better: 0.12
10: AVG Simple: 6.88
10: AVG Breakpoint: 6.94

25: Breakpoints Better: 0.23
25: AVG Simple: 21.39
25: AVG Breakpoint: 21.45



## Assignment 3

The knapsack problem is a combinatorial optimisation problem. Given as input is a set of objects $i_1, \ldots, i_n$ and a weight bound $G\in\mathbb{R}_{\geq0}$. Each item $i_j$ has a weight $w_j \in \mathbb{R}_{\geq0}$ and a value $v_j \in \mathbb{R}_{\geq0}$. We are looking for a subset $S \subseteq \{i_1,\ldots, i_n\}$ of the objects so that their total weight is at most $G$, i.e. $\sum_{i_j \in S} w_j \leq G$, and their value $\sum_{i_j \in S} v_j$ is maximised.

The problem can be solved with a greedy algorithm that sorts all items in descending order according to their ratio of value to weight. An order $k_1, \ldots, k_n$ of the items is thus formed so that

$$\frac{v_{k_1}}{w_{k_1}} \geq \frac{v_{k_2}}{w_{k_2}} \geq \ldots \geq \frac{v_{k_n}}{w_{k_n}}$$

This means that $i_{k_1}$ is the item with the best ratio of value to weight and $i_{k_n}$ is the item with the worst ratio. The items are then placed in the knapsack in this order, provided it does not become too heavy. If an item no longer fits, it is skipped and the next item is added until the last item has been checked.

1. Implement this greedy algorithm.

In [ ]:
def greedy_knapsack(weights, values, limit):
    # weights and values are lists of size n containing numeric values
    # output should be a sorted list of item indices (ranging from 0 to n-1)
    pass

In [ ]:
test(greedy_knapsack([1, 1, 1], [3, 7, 5], 2), [1, 2])
test(greedy_knapsack([1, 1, 3], [3, 5, 7], 3), [0, 1])
test(greedy_knapsack([1, 2, 3, 4], [2, 5, 7, 8], 4), [0, 1])
test(greedy_knapsack([4, 5, 6, 7], [5, 6, 7, 8], 15), [0, 1, 2])
test(greedy_knapsack([7, 5, 6, 4], [8, 6, 7, 5], 14), [1, 3])
test(greedy_knapsack([7, 5, 6, 4, 4], [8, 6, 7, 5, 4], 14), [1, 3, 4])
test(greedy_knapsack([7, 5, 6, 4, 4], [8, 6, 7, 5, 4], 15), [1, 2, 3])
test(greedy_knapsack([5, 8, 10], [3, 4, 5], 4), [])

2. What approximation ratio does this algorithm have? Give reasons for your answer.